In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier, export_text

# 1. CARREGAR A BASE DE DADOS
url = "https://docs.google.com/spreadsheets/d/1g1aQ61vijh6uHJuc8sijeBEMsoIQ2a5yLwUK04Wptlg/export?format=csv"
df = pd.read_csv(url)

# Mapeamento para nomes simples e sem espaços
mapping = {
    "Carimbo de data/hora": "timestamp",
    "Você ficou gripado no ano passado ?": "gripe_ano_passado",
    "Você tomou vacina da gripe no ano passado?": "vacina",
    "  Você frequentou no ano passado,  semanalmente ambientes com muitas pessoas? (salas cheias, ônibus, eventos, etc.)  ": "ambientes_cheios",
    "  Você viajou no ano passado mais de 100 km de distância?  ": "viajou",
    "  Você tem alergia nas vias aéreas (rinite, sinusite, etc.)?  ": "alergia",
    "Quantas horas você dormiu em média por noite no ano passado?": "horas_sono",
    "Você praticou atividade física no ano passado?": "exercicio",
    "Você se alimentou de forma balanceada no ano passado?": "alimentacao",
    "Em média, quantas vezes você lavou as mãos por dia no ano passado?": "lavagem_maos",
    "Na sua percepção, o seu nível de estresse no ano passado foi:": "estresse"
}

# Renomear as colunas e remover a coluna de data/hora
df = df.rename(columns=mapping)
df = df.drop(columns=['timestamp'])

# Preencher o único valor de estresse vazio com a mediana
df['estresse'] = df['estresse'].fillna(df['estresse'].median())

# Separar características (X) do alvo que queremos prever (y)
X = df.drop(columns=['gripe_ano_passado'])
y = df['gripe_ano_passado']

X = pd.get_dummies(X, drop_first=True)
le = LabelEncoder()
y = le.fit_transform(y) # Transforma "Não" em 0 e "Sim" em 1

# Divisão de treino (80%) e teste (20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ==========================================
# ALGORITMO: ÁRVORE DE DECISÃO & REGRAS
# ==========================================

# 1. Treinamos a Árvore de Decisão
# Limitamos a profundidade (max_depth=3) para as regras não ficarem gigantes e confusas
dt = DecisionTreeClassifier(random_state=42, max_depth=3)
dt.fit(X_train, y_train)

# 2. Extraímos as Regras da Árvore
regras = export_text(dt, feature_names=list(X.columns))

print("="*50)
print(" SISTEMA DE REGRAS EXTRAÍDO DA ÁRVORE ")
print("="*50)
print("Guia de Leitura:")
print(" - Se a condição for '<= 0.50', significa que a resposta do usuário foi 'Não/Falso'.")
print(" - Se a condição for '>  0.50', significa que a resposta do usuário foi 'Sim/Verdadeiro'.")
print(" - class: 1 significa 'Ficou Gripado'")
print(" - class: 0 significa 'Não Ficou Gripado'")
print("-" * 50)
print(regras)

 SISTEMA DE REGRAS EXTRAÍDO DA ÁRVORE 
Guia de Leitura:
 - Se a condição for '<= 0.50', significa que a resposta do usuário foi 'Não/Falso'.
 - Se a condição for '>  0.50', significa que a resposta do usuário foi 'Sim/Verdadeiro'.
 - class: 1 significa 'Ficou Gripado'
 - class: 0 significa 'Não Ficou Gripado'
--------------------------------------------------
|--- lavagem_maos_Mais de 10 vezes <= 0.50
|   |--- viajou_Pelo menos uma vez por mês <= 0.50
|   |   |--- alimentacao_Sim, a maior parte do tempo <= 0.50
|   |   |   |--- class: 1
|   |   |--- alimentacao_Sim, a maior parte do tempo >  0.50
|   |   |   |--- class: 1
|   |--- viajou_Pelo menos uma vez por mês >  0.50
|   |   |--- alergia_Médio <= 0.50
|   |   |   |--- class: 0
|   |   |--- alergia_Médio >  0.50
|   |   |   |--- class: 1
|--- lavagem_maos_Mais de 10 vezes >  0.50
|   |--- estresse <= 4.50
|   |   |--- estresse <= 3.50
|   |   |   |--- class: 0
|   |   |--- estresse >  3.50
|   |   |   |--- class: 1
|   |--- estress